# Capstone Two — Pre-processing and Training Data Development
**Project:** Predicting Residential Solar Adoption Across U.S. Counties
**Author:** Colin Segura — Springboard Data Science Career Track

This notebook covers Step 4 of the capstone: turning the cleaned, EDA-vetted
county dataset into a model-ready train/test split. The work happens in five
ordered operations:

1. **Handle missing values** — median imputation with missing-indicator flags
2. **Engineer features** — log transforms, derived variables (`housing_age`)
3. **Encode categorical variables** — one-hot encoding for `state`
4. **Build feature matrix and target** — drop leakage and collinear columns
5. **Train/test split, then standardize numeric features** — fit the scaler
   on training data only, then apply to both train and test

The output is four CSVs (`X_train_scaled.csv`, `X_test_scaled.csv`,
`y_train.csv`, `y_test.csv`) that the modeling notebook will read in
directly.

> **Order matters.** Scaling happens *after* the train/test split, not
> before. Fitting a scaler on the full dataset before splitting would leak
> information from the test set into the training set. The same logic
> would apply to imputation in a strict production pipeline; here we
> impute before splitting because the missingness is tiny (3 of 50 rows,
> all from known Census suppression codes), but I'll note the more
> rigorous approach in the closing comments.


## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Load the cleaned dataset produced by the wrangling notebook
df = pd.read_csv("county_solar_adoption.csv", dtype={"county_fips": "string"})
print(f"Loaded shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Loaded shape: (50, 15)
Columns: ['county_fips', 'county_name', 'state', 'installs_total', 'installed_kw_total', 'installs_per_1k_oo_hh', 'kw_per_1k_oo_hh', 'median_household_income', 'owner_occupied_units', 'total_housing_units', 'median_year_built', 'population', 'residential_price_cents_kwh', 'median_system_size_kw', 'median_cost_per_watt']


## 1. Handle missing values

The EDA committed to median imputation with a binary missing-indicator
column. Median (rather than mean) because both affected features
(`median_household_income`, `median_year_built`) are right-skewed in
real Census data and the median is more robust. The missing-indicator
column lets the model learn whether the suppression itself was
predictive, which it often is — suppressed-estimate counties tend to be
smaller and less urban.


In [2]:
# Check the missingness situation before imputing
missing_before = df.isna().sum()
print("Missing values per column (before imputation):")
print(missing_before[missing_before > 0].to_string())
print(f"\nRows with any missing value: {df.isna().any(axis=1).sum()} of {len(df)}")


Missing values per column (before imputation):
median_household_income    2
median_year_built          1

Rows with any missing value: 3 of 50


In [3]:
# Add missing-indicator columns for the two affected features
df["income_was_missing"] = df["median_household_income"].isna().astype(int)
df["year_built_was_missing"] = df["median_year_built"].isna().astype(int)

# Median impute the affected columns
imputer = SimpleImputer(strategy="median")
df[["median_household_income", "median_year_built"]] = imputer.fit_transform(
    df[["median_household_income", "median_year_built"]]
)

print("Missing values per column (after imputation):")
remaining = df.isna().sum()
print(remaining[remaining > 0].to_string() if (remaining > 0).any() else "  (none)")

print(f"\nNew columns:")
print(f"  income_was_missing      flagged {df['income_was_missing'].sum()} rows")
print(f"  year_built_was_missing  flagged {df['year_built_was_missing'].sum()} rows")


Missing values per column (after imputation):
  (none)

New columns:
  income_was_missing      flagged 2 rows
  year_built_was_missing  flagged 1 rows


## 2. Engineer features

These transformations were planned in the EDA. None of them depend on the
train/test split, so they're applied to the full dataset before splitting.

- **`log_population`** — population is heavily right-skewed (LA County to
  small Connecticut counties in one column), so a log transform makes
  linear-model coefficients more interpretable.
- **`housing_age`** — current year minus median year built. More
  intuitive than the raw year column and gives a directly interpretable
  effect size ("each additional year of housing age").
- **`log1p_target`** — log(1 + target). Available as an alternative
  target for linear models; the EDA showed strong right-skew on the raw
  target (skewness 2.4).


In [4]:
# Apply the engineered features
df["log_population"] = np.log(df["population"])
df["housing_age"] = 2024 - df["median_year_built"]
df["log1p_target"] = np.log1p(df["installs_per_1k_oo_hh"])

print("Engineered features sample:")
print(df[["county_name", "state", "population", "log_population",
          "median_year_built", "housing_age",
          "installs_per_1k_oo_hh", "log1p_target"]].head(8).to_string())


Engineered features sample:
            county_name state  population  log_population  median_year_built  housing_age  installs_per_1k_oo_hh  log1p_target
0    Los Angeles County    CA   1787233.0       14.396179             1982.0         42.0               1.250718      0.811249
1         Orange County    CA    229538.0       12.343824             1979.0         45.0               6.874642      2.063648
2      San Diego County    CA   1302127.0       14.079510             1988.0         36.0               0.849654      0.614999
3  San Francisco County    CA   2141652.0       14.577088             1963.0         61.0               0.577505      0.455844
4      San Mateo County    CA   2404333.0       14.692783             1978.0         46.0               1.042154      0.714005
5    Santa Clara County    CA   1450351.0       14.187316             1988.0         36.0               1.501228      0.916782
6       Maricopa County    AZ    667641.0       13.411506             1973.0       

## 3. One-hot encode the categorical feature

The dataset has one categorical column for modeling: `state`. The EDA was
ambivalent about including state directly (the ANOVA didn't formally
detect a state effect because of small sample size per state, but the
boxplot showed visible state-level differences). The pragmatic answer is
to include the one-hot encoded columns as optional features so the model
can use them if they help.

**Notes on dimensionality.** With 50 rows and 21 states, one-hot
encoding state adds 20 columns (drop_first=True to avoid the dummy
variable trap). That's a lot of features relative to row count, and the
modeling step may need regularization or feature selection to manage
that. For the real ~3,100-county dataset, this won't be an issue.


In [5]:
# pandas.get_dummies with drop_first=True for the standard reference-encoding
state_dummies = pd.get_dummies(df["state"], prefix="state", drop_first=True, dtype=int)
print(f"Created {state_dummies.shape[1]} state dummy columns (dropped one as reference)")
print(f"Columns: {list(state_dummies.columns)}")

# Concatenate the dummies into the main dataframe
df = pd.concat([df, state_dummies], axis=1)
print(f"\nDataframe shape after adding dummies: {df.shape}")


Created 20 state dummy columns (dropped one as reference)
Columns: ['state_CA', 'state_CO', 'state_CT', 'state_FL', 'state_GA', 'state_IL', 'state_MA', 'state_MD', 'state_MI', 'state_NC', 'state_NJ', 'state_NV', 'state_NY', 'state_OH', 'state_OR', 'state_PA', 'state_TX', 'state_UT', 'state_VA', 'state_WA']

Dataframe shape after adding dummies: (50, 40)


## 4. Build feature matrix X and target y

Now we explicitly select the feature columns and the target, dropping
the columns the EDA flagged as either leakage or collinear.

**Drop (leakage):** `installs_total`, `installed_kw_total`,
`kw_per_1k_oo_hh`. These are either pre-aggregation versions of the
target or alternate forms of it; including them would let the model
trivially predict perfectly.

**Drop (collinear):** `owner_occupied_units`, `total_housing_units`.
Both are extremely highly correlated with `population` (r ≈ 0.9+) and
have already done their work as denominators when building the per-capita
target. Keeping any of them would create unstable coefficients.

**Drop (identifiers):** `county_fips`, `county_name`. Identifiers, not
features.

**Drop (raw versions of engineered features):** `population`,
`median_year_built`. Replaced by `log_population` and `housing_age`.

**Drop (raw categorical, replaced by one-hot):** `state`.


In [6]:
# Define the target — primary target is raw, alternate is log-transformed
y = df["installs_per_1k_oo_hh"].copy()
y_log = df["log1p_target"].copy()

# Define the feature matrix by explicit selection
feature_cols = [
    # Continuous numeric features
    "median_household_income",
    "log_population",
    "housing_age",
    "residential_price_cents_kwh",
    "median_system_size_kw",
    "median_cost_per_watt",
    # Missing-indicator flags
    "income_was_missing",
    "year_built_was_missing",
] + list(state_dummies.columns)  # one-hot state columns

X = df[feature_cols].copy()
print(f"Feature matrix X shape: {X.shape}")
print(f"Target y shape: {y.shape}")
print(f"\nFeature columns ({len(feature_cols)} total):")
for c in feature_cols:
    print(f"  {c}")


Feature matrix X shape: (50, 28)
Target y shape: (50,)

Feature columns (28 total):
  median_household_income
  log_population
  housing_age
  residential_price_cents_kwh
  median_system_size_kw
  median_cost_per_watt
  income_was_missing
  year_built_was_missing
  state_CA
  state_CO
  state_CT
  state_FL
  state_GA
  state_IL
  state_MA
  state_MD
  state_MI
  state_NC
  state_NJ
  state_NV
  state_NY
  state_OH
  state_OR
  state_PA
  state_TX
  state_UT
  state_VA
  state_WA


## 5. Train/test split

An 80/20 split with a fixed `random_state=42` for reproducibility. With
50 rows, that gives 40 training and 10 testing observations — small,
but it's what the synthetic sample supports. On the real ~3,100-county
dataset the same proportions give a much more comfortable ~620-row test
set.

The split is on `(X, y)` together so that the same county appears in
the same partition across the feature matrix and the target. We also
split `y_log` against the same indices so the log-transformed target
stays aligned with the raw one.


In [7]:
# Single split with reproducible random_state
X_train, X_test, y_train, y_test, y_log_train, y_log_test = train_test_split(
    X, y, y_log,
    test_size=0.20,
    random_state=42,
)

print(f"Training set:  X_train {X_train.shape},  y_train {y_train.shape}")
print(f"Testing set:   X_test  {X_test.shape},   y_test  {y_test.shape}")

print(f"\nTarget statistics:")
print(f"  y_train mean: {y_train.mean():.3f},  std: {y_train.std():.3f}")
print(f"  y_test  mean: {y_test.mean():.3f},  std: {y_test.std():.3f}")


Training set:  X_train (40, 28),  y_train (40,)
Testing set:   X_test  (10, 28),   y_test  (10,)

Target statistics:
  y_train mean: 0.954,  std: 1.258
  y_test  mean: 1.934,  std: 1.967


## 6. Standardize numeric features

`StandardScaler` rescales each feature to have mean 0 and standard
deviation 1. This matters for linear models (gradient descent
convergence, interpretable coefficients), distance-based models (KNN,
SVM), and any L1/L2 regularization where unscaled features would be
penalized unfairly. Tree-based models don't need scaling, but applying
it consistently doesn't hurt them.

**Critical detail:** the scaler is `fit` on the training set only, then
used to `transform` both training and test sets. Fitting on the combined
dataset would leak test-set statistics into training. The dummy /
indicator columns are already 0/1 and don't need scaling, but I'll
include them in the scaler for tidiness — scaling a 0/1 column just
re-centers it to roughly ±1.


In [8]:
# Identify which columns to scale: every column in X (we'll scale them all
# uniformly so the resulting X_train_scaled has the same column ordering)
numeric_cols = X.columns.tolist()

scaler = StandardScaler()
scaler.fit(X_train[numeric_cols])  # fit on TRAIN only

X_train_scaled = pd.DataFrame(
    scaler.transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index,
)

print(f"Scaled training set shape: {X_train_scaled.shape}")
print(f"Scaled testing set shape:  {X_test_scaled.shape}")

print(f"\nSanity check — training set scaling:")
print(f"  Mean of scaled features (should be ~0):")
print(X_train_scaled.mean().abs().describe().round(4).to_string())
print(f"  Std of scaled features (should be ~1):")
print(X_train_scaled.std().describe().round(4).to_string())

print(f"\nSanity check — test set scaling (NOT centered exactly, as expected):")
print(f"  Mean of scaled test features (small but nonzero, since fitted on train):")
print(X_test_scaled.mean().abs().describe().round(4).to_string())

# Diagnostic: identify any zero-variance columns in training
# (happens when a state's counties all land in one partition after splitting)
train_var = X_train[numeric_cols].var()
zero_var_cols = train_var[train_var == 0].index.tolist()
if zero_var_cols:
    print(f"\nZero-variance columns in training set ({len(zero_var_cols)}):")
    for c in zero_var_cols:
        # Which state and where its counties ended up
        if c.startswith("state_"):
            state_code = c.replace("state_", "")
            in_train = (X_train[c] == 1).sum()
            in_test = (X_test[c] == 1).sum()
            print(f"  {c:25s}  train: {in_train} rows, test: {in_test} rows")
        else:
            print(f"  {c}")
    print("""
  Note: this is a small-sample artifact. With only 40 training rows across
  21 states, some states unavoidably have all their counties in one
  partition. sklearn's StandardScaler handles zero-variance columns
  gracefully by setting scale=1, so the column simply becomes uninformative
  in training. On the full ~3,100-county dataset this won't happen.""")
else:
    print("\nNo zero-variance columns in training set.")


Scaled training set shape: (40, 28)
Scaled testing set shape:  (10, 28)

Sanity check — training set scaling:
  Mean of scaled features (should be ~0):
count    28.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
  Std of scaled features (should be ~1):
count    28.0000
mean      0.9766
std       0.1914
min       0.0000
25%       1.0127
50%       1.0127
75%       1.0127
max       1.0127

Sanity check — test set scaling (NOT centered exactly, as expected):
  Mean of scaled test features (small but nonzero, since fitted on train):
count    28.0000
mean      0.3467
std       0.2381
min       0.0949
25%       0.2160
50%       0.2294
75%       0.4804
max       1.1209

Zero-variance columns in training set (1):
  state_NV                   train: 0 rows, test: 1 rows

  Note: this is a small-sample artifact. With only 40 training rows across
  21 states, some states unavoidably have all their counties in one
  partition. sklearn's StandardSc

## 7. Save the train/test splits

Saving as four CSVs makes them straightforward to load in the modeling
notebook without re-running this entire pipeline:

- `X_train_scaled.csv`, `X_test_scaled.csv` — scaled feature matrices
- `y_train.csv`, `y_test.csv` — raw and log-transformed targets paired

Index is preserved so individual counties can be traced back to the
original dataframe for error analysis.


In [9]:
# Build target dataframes that carry both the raw and log targets
y_train_df = pd.DataFrame({
    "installs_per_1k_oo_hh": y_train,
    "log1p_target": y_log_train,
})
y_test_df = pd.DataFrame({
    "installs_per_1k_oo_hh": y_test,
    "log1p_target": y_log_test,
})

# Write all four files
X_train_scaled.to_csv("X_train_scaled.csv", index=True, index_label="row_id")
X_test_scaled.to_csv("X_test_scaled.csv", index=True, index_label="row_id")
y_train_df.to_csv("y_train.csv", index=True, index_label="row_id")
y_test_df.to_csv("y_test.csv", index=True, index_label="row_id")

import os
print("Saved files (size in KB):")
for name in ["X_train_scaled.csv", "X_test_scaled.csv", "y_train.csv", "y_test.csv"]:
    size_kb = os.path.getsize(name) / 1024
    print(f"  {name:25s}  {size_kb:6.1f} KB")


Saved files (size in KB):
  X_train_scaled.csv           22.0 KB
  X_test_scaled.csv             5.8 KB
  y_train.csv                   1.6 KB
  y_test.csv                    0.4 KB


## 8. Summary and notes for the next step

**What this notebook produced:**

- A model-ready feature matrix with 8 base features plus 20 one-hot
  state indicators = 28 columns total
- A separate raw target and log-transformed target, paired by index
- Train/test split: 40 training counties / 10 testing counties
  (will be ~2,500 / ~620 on the real dataset)
- StandardScaler fitted on training data only, applied to both sets

**Decisions worth flagging:**

- **Imputation before split.** I imputed missing values on the full
  dataset before splitting. With only 3 rows missing values, the
  information leak is negligible. The strict production pattern is to
  put imputation inside an `sklearn.pipeline.Pipeline` along with the
  scaler so both fit on train only. Worth implementing for the real
  dataset where there will be more missingness.
- **State dummies included.** This adds 20 columns to a 50-row dataset.
  The modeling step may need regularization (Ridge / Lasso) or feature
  selection to manage that dimensionality. On the full dataset the
  problem disappears.
- **All columns scaled uniformly.** Including the 0/1 missing-indicator
  and state-dummy columns through the scaler is harmless but not
  strictly necessary. A `ColumnTransformer` would let us scale only the
  truly continuous features. Defer to mentor preference.

**Next step:** Modeling. Load the four CSVs, fit a baseline linear
regression on the scaled features against the log-transformed target,
record the metrics, then move to a gradient-boosted tree model on the
raw target with cross-validation. The interpretation will rely on
feature importance and SHAP values, as planned in the proposal.

**Open questions for the mentor call:**

- Confirm 80/20 split ratio is appropriate, or whether to consider
  stratified splits by state to ensure all states appear in training.
- Whether to standardize the target (raw or log) as well — most
  regression workflows leave the target in natural units for
  interpretability, but some prefer scaling everything.
- Whether to keep the `state` dummies given the dimensionality concern,
  or rely on state-level drivers (price, future policy indicators)
  alone.
